# ACE-Net Universal Shard Preprocessor
### Assigned Member: **JC**
### Target Account / Google Profile: `promptingacc@gmail.com`
### Target Dataset: **TRACK_3** | Shard: **`shard_0003`**
### Output Target: `Google Drive > THESIS_MOTHERFILE > Baseline preprocessed > TRACK_3`

## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Step 2: Clone Repository & Checkout Branch

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!git log --oneline -1

## Step 3: Install Required Dependencies

In [ ]:
!pip install -q --no-deps facenet-pytorch
!pip install -q --no-deps git+https://github.com/openai/whisper.git
print('Dependencies installed successfully!')

## Step 4: Fast Unzip Raw Dataset (`tracks_1_2_3_4.zip`) to Local Colab SSD

In [ ]:
import os

DRIVE_ZIP = '/content/drive/MyDrive/THESIS_MOTHERFILE/datasets/tracks_1_2_3_4.zip'
LOCAL_RAW = '/content/data/raw/TRACK_3'

os.makedirs(LOCAL_RAW, exist_ok=True)
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f'Raw zip not found in Drive: {DRIVE_ZIP}')

print(f'Fast unzipping {DRIVE_ZIP} to local SSD ({LOCAL_RAW})...')
!unzip -q -n '{DRIVE_ZIP}' -d '{LOCAL_RAW}'
print('Unzip complete! Local files ready.')

## Step 5: Execute Preprocessing for `TRACK_3` [shard_0003]

In [ ]:
!python scripts/preprocess/run_shard.py \
    --account 'promptingacc@gmail.com' \
    --dataset 'TRACK_3' \
    --shard '0003' \
    --raw_dir '/content/data/raw/TRACK_3' \
    --drive_root '/content/drive/MyDrive/THESIS_MOTHERFILE' \
    --device cuda

## Step 6: Post-Run Integrity Check & Verification

In [ ]:
import json, glob
from pathlib import Path

ckpt_file = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline preprocessed/TRACK_3/checkpoints/shard_0003_checkpoint.json')
out_shard = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline preprocessed/TRACK_3/shards/shard_0003')

if ckpt_file.exists():
    with open(ckpt_file) as f:
        data = json.load(f)
    print('=' * 60)
    print('SHARD STATUS:', data.get('status'))
    print('Completed Clips:', len(data.get('completed_ids', [])))
    print('Failed Clips   :', len(data.get('failed_ids', [])))
    print('=' * 60)
else:
    print('Checkpoint not found. Run Step 5 first.')